Import Library



In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import RobustScaler
from sklearn.ensemble import ExtraTreesRegressor
from sklearn.linear_model import ElasticNet
from sklearn.svm import SVR
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

pd.set_option('display.max_columns', None)

Load Data & Preprocessing Dataset


In [12]:
# 1. Load the dataset
df = pd.read_csv('ispu_dki1.csv')

# 2. Parse the dates and extract the year
df['tanggal'] = pd.to_datetime(df['tanggal'])
df['tahun'] = df['tanggal'].dt.year

features = ['pm10', 'so2', 'co', 'o3', 'no2']
target = 'pm25'

# 3. Convert the feature and target columns to numeric values
for col in features + [target]:
    df[col] = pd.to_numeric(df[col], errors='coerce')

print(f"Total rows in the original dataset: {len(df)}")
print(f"Data year range: {df['tahun'].min()} - {df['tahun'].max()}")

Total rows in the original dataset: 5173
Data year range: 2010 - 2025


In [14]:

import pandas as pd

# Load the original dataset
df = pd.read_csv("ispu_dki1.csv")

# Parse the dates and convert the relevant columns to numeric values
df["tanggal"] = pd.to_datetime(df["tanggal"], errors="coerce")
features = ["pm10", "so2", "co", "o3", "no2"]
target = "pm25"

for col in features + [target]:
    df[col] = pd.to_numeric(df[col], errors="coerce")

# Keep records from 2010–2020
historical = df[
    df["tanggal"].dt.year.between(2010, 2020)
].copy()

# Check data completeness
pm25_missing = historical[target].isna()
features_complete = historical[features].notna().all(axis=1)

total_rows = len(historical)
missing_pm25 = int(pm25_missing.sum())
predictable_rows = int((pm25_missing & features_complete).sum())
missing_features = int((pm25_missing & ~features_complete).sum())

print("=== DATA CHECK: 2010–2020 ===")
print("Total historical rows             :", total_rows)
print("Missing PM2.5 values               :", missing_pm25)
print("Rows eligible for estimation      :", predictable_rows)
print("Rows with incomplete predictors   :", missing_features)
print("Total rows in the CSV              :", len(df))

# Count missing values in each feature and target column
print("\n=== MISSING VALUES BY COLUMN ===")
print(historical[features + [target]].isna().sum())


=== DATA CHECK: 2010–2020 ===
Total historical rows             : 4018
Missing PM2.5 values               : 4018
Rows eligible for estimation      : 3585
Rows with incomplete predictors   : 433
Total rows in the CSV              : 5173

=== MISSING VALUES BY COLUMN ===
pm10     212
so2      175
co       150
o3       203
no2      193
pm25    4018
dtype: int64


Train and Evaluate Three Models (2021–2025 Data)


Methodology note: The models are evaluated using a chronological split: earlier observations are used for training and later observations for testing. The CSV file has no records for 2022, and actual PM2.5 values are missing throughout 2010–2020. Therefore, PM2.5 values reconstructed for 2010–2020 should be treated as estimates, and this limitation should be discussed in the conclusion.

In [17]:

# Make sure the date column is parsed as datetime
df['tanggal'] = pd.to_datetime(df['tanggal'], errors='coerce')

# Extract the year from each date
df['tahun'] = df['tanggal'].dt.year

# Check the parsed dates and extracted years
print(df[['tanggal', 'tahun']].head())


     tanggal  tahun
0 2010-01-01   2010
1 2010-01-02   2010
2 2010-01-03   2010
3 2010-01-04   2010
4 2010-01-05   2010


In [18]:
# 1. Select complete records for model evaluation (2021–2025)
df_2021_2025 = df[
    (df['tahun'] >= 2021) &
    (df['tahun'] <= 2025) &
    (df[target].notna()) &
    (df[features].notna().all(axis=1))
].copy()

# Keep the complete recent data for retraining the selected model later
X_recent = df_2021_2025[features]
y_recent = df_2021_2025[target]

# Sort by date so the test set represents a later time period
df_2021_2025 = df_2021_2025.sort_values('tanggal').reset_index(drop=True)

# 2. Split by date: use the earliest 80% of dates for training and the latest 20% for testing
unique_dates = np.sort(df_2021_2025['tanggal'].dropna().unique())
if len(unique_dates) < 2:
    raise ValueError("There are not enough complete records to create a time-based train/test split.")

split_idx = int(len(unique_dates) * 0.8)
split_idx = min(max(split_idx, 1), len(unique_dates) - 1)
split_date = pd.Timestamp(unique_dates[split_idx])

train_data = df_2021_2025[df_2021_2025['tanggal'] < split_date].copy()
test_data = df_2021_2025[df_2021_2025['tanggal'] >= split_date].copy()

X_tr = train_data[features]
y_tr = train_data[target]
X_val = test_data[features]
y_val = test_data[target]

print("========== CHRONOLOGICAL TRAIN/TEST SPLIT ==========")
print(f"Training period: {train_data['tanggal'].min().date()} to {train_data['tanggal'].max().date()}")
print(f"Testing period : {test_data['tanggal'].min().date()} to {test_data['tanggal'].max().date()}")
print(f"Training rows  : {len(train_data)}")
print(f"Testing rows   : {len(test_data)}")

# 3. Define the models. Apply feature scaling to ElasticNet and SVR only.
models = {
    'Extra Trees Regressor': {
        'model': ExtraTreesRegressor(n_estimators=100, random_state=42),
        'scaled': False
    },
    'ElasticNet Regression': {
        'model': ElasticNet(alpha=0.1, l1_ratio=0.5, random_state=42),
        'scaled': True
    },
    'Support Vector Regressor (SVR)': {
        'model': SVR(C=10.0, epsilon=0.1, kernel='rbf'),
        'scaled': True
    }
}

def index_of_agreement(y_true, y_pred):
    y_true = np.asarray(y_true, dtype=float)
    y_pred = np.asarray(y_pred, dtype=float)
    y_mean = np.mean(y_true)
    numerator = np.sum((y_true - y_pred) ** 2)
    denominator = np.sum(
        (np.abs(y_pred - y_mean) + np.abs(y_true - y_mean)) ** 2
    )
    if denominator == 0:
        return np.nan
    return 1 - numerator / denominator

# 4. Train and evaluate all models on the same test period
results = []
trained_models = {}
scalers = {}

for name, config in models.items():
    model = config['model']

    if config['scaled']:
        scaler_model = RobustScaler()
        X_train_model = scaler_model.fit_transform(X_tr)
        X_test_model = scaler_model.transform(X_val)
        scalers[name] = scaler_model
    else:
        X_train_model = X_tr
        X_test_model = X_val

    model.fit(X_train_model, y_tr)
    y_pred = model.predict(X_test_model)
    trained_models[name] = model

    results.append({
        'Model': name,
        'R2 Score': r2_score(y_val, y_pred),
        'RMSE': np.sqrt(mean_squared_error(y_val, y_pred)),
        'MAE': mean_absolute_error(y_val, y_pred),
        'Index of Agreement (d)': index_of_agreement(y_val, y_pred)
    })

df_results = pd.DataFrame(results).sort_values('RMSE').reset_index(drop=True)
print("\n========== COMPARISON OF THREE MODELS ==========")
print(df_results.round(4).to_string(index=False))

# Select the model with the lowest RMSE
best_model_name = df_results.loc[0, 'Model']
print(f"\nBest model based on RMSE: {best_model_name}")
print("Note: this evaluation measures performance on a later period; it does not guarantee reconstruction accuracy for 2010–2020.")


========== CHRONOLOGICAL TRAIN/TEST SPLIT ==========
Training period: 2021-01-01 to 2024-07-11
Testing period : 2024-07-12 to 2025-02-28
Training rows  : 899
Testing rows   : 225

========== COMPARISON OF THREE MODELS ==========
                         Model  R2 Score   RMSE    MAE  Index of Agreement (d)
         ElasticNet Regression    0.8664 7.4709 5.0690                  0.9628
         Extra Trees Regressor    0.8526 7.8500 5.5141                  0.9579
Support Vector Regressor (SVR)    0.7635 9.9425 6.7704                  0.9118

Best model based on RMSE: ElasticNet Regression
Note: this evaluation measures performance on a later period; it does not guarantee reconstruction accuracy for 2010–2020.


Reconstruct PM2.5 for 2010–2020 Using the Best Model

In [19]:
# Retrain the selected model using all available records with actual PM2.5 values (2021–2025)
# The selected model is based on the chronological evaluation in the previous cell.
best_config = models[best_model_name]
best_model = best_config['model']

if best_config['scaled']:
    reconstruction_scaler = RobustScaler()
    X_recent_final = reconstruction_scaler.fit_transform(X_recent)
else:
    reconstruction_scaler = None
    X_recent_final = X_recent

best_model.fit(X_recent_final, y_recent)

# Keep the original PM2.5 values and store estimates in a separate column
df['pm25_reconstructed'] = df[target]

# Reconstruct values only for 2010–2020 rows where PM2.5 is missing and all predictor features are available
reconstruct_mask = (
    (df['tahun'] >= 2010) & (df['tahun'] <= 2020) &
    (df[target].isna()) &
    (df[features].notna().all(axis=1))
)

X_reconstruct = df.loc[reconstruct_mask, features]
if len(X_reconstruct) > 0:
    if best_config['scaled']:
        X_reconstruct_model = reconstruction_scaler.transform(X_reconstruct)
    else:
        X_reconstruct_model = X_reconstruct

    predictions = best_model.predict(X_reconstruct_model)

    # PM2.5 concentrations cannot be negative
    df.loc[reconstruct_mask, 'pm25_reconstructed'] = np.clip(predictions, 0, None)

# Summarize the reconstruction results for 2010–2020
df_2010_2020_raw = df[(df['tahun'] >= 2010) & (df['tahun'] <= 2020)]
total_missing_2010_2020 = df_2010_2020_raw[target].isna().sum()
reconstructed_count = int(reconstruct_mask.sum())
remaining_missing = df.loc[
    (df['tahun'] >= 2010) & (df['tahun'] <= 2020),
    'pm25_reconstructed'
].isna().sum()

print("========== PM2.5 RECONSTRUCTION REPORT (2010–2020) ==========")
print(f"Model used                         : {best_model_name}")
print(f"Total rows from 2010–2020          : {len(df_2010_2020_raw)}")
print(f"Initially missing PM2.5 values     : {total_missing_2010_2020}")
print(f"Rows successfully estimated        : {reconstructed_count}")
print(f"PM2.5 values still missing         : {remaining_missing}")
print("Important: reconstructed values are model estimates, not actual PM2.5 measurements.")


========== PM2.5 RECONSTRUCTION REPORT (2010–2020) ==========
Model used                         : ElasticNet Regression
Total rows from 2010–2020          : 4018
Initially missing PM2.5 values     : 4018
Rows successfully estimated        : 3585
PM2.5 values still missing         : 433
Important: reconstructed values are model estimates, not actual PM2.5 measurements.


Calculate ISPU Sub-Indices (Minister of Environment and Forestry Regulation No. 14/2020)

In [21]:
BREAKPOINTS = {
    'pm25': [(0, 50, 0.0, 15.5), (51, 100, 15.6, 55.4), (101, 200, 55.5, 150.4), (201, 300, 150.5, 250.4), (301, 500, 250.5, 500.0)],
    'pm10': [(0, 50, 0.0, 50.0), (51, 100, 51.0, 150.0), (101, 200, 151.0, 350.0), (201, 300, 351.0, 420.0), (301, 500, 421.0, 500.0)],
    'so2':  [(0, 50, 0.0, 52.0), (51, 100, 53.0, 180.0), (101, 200, 181.0, 400.0), (201, 300, 401.0, 800.0), (301, 500, 801.0, 1200.0)],
    'co':   [(0, 50, 0.0, 4000.0), (51, 100, 4001.0, 8000.0), (101, 200, 8001.0, 15000.0), (201, 300, 15001.0, 30000.0), (301, 500, 30001.0, 45000.0)],
    'o3':   [(0, 50, 0.0, 120.0), (51, 100, 121.0, 235.0), (101, 200, 236.0, 400.0), (201, 300, 401.0, 800.0), (301, 500, 801.0, 1000.0)],
    'no2':  [(0, 50, 0.0, 80.0), (51, 100, 81.0, 200.0), (101, 200, 201.0, 1130.0), (201, 300, 1131.0, 2260.0), (301, 500, 2261.0, 3000.0)]
}

def calc_subindex(conc, pollutant):
    if pd.isna(conc) or pollutant not in BREAKPOINTS:
        return np.nan
    for Ia, Ib, Xa, Xb in BREAKPOINTS[pollutant]:
        if Xa <= conc <= Xb:
            return ((Ib - Ia) / (Xb - Xa)) * (conc - Xa) + Ia
    if conc > BREAKPOINTS[pollutant][-1][3]:
        return 500.0
    return np.nan

for p in ['pm10', 'so2', 'co', 'o3', 'no2']:
    df[f'ispu_{p}'] = df[p].apply(lambda x: calc_subindex(x, p))

df['ispu_pm25_raw'] = df[target].apply(lambda x: calc_subindex(x, 'pm25'))
df['ispu_pm25_rec'] = df['pm25_reconstructed'].apply(lambda x: calc_subindex(x, 'pm25'))

ispu_cols_before = ['ispu_pm10', 'ispu_so2', 'ispu_co', 'ispu_o3', 'ispu_no2', 'ispu_pm25_raw']
ispu_cols_after  = ['ispu_pm10', 'ispu_so2', 'ispu_co', 'ispu_o3', 'ispu_no2', 'ispu_pm25_rec']

df['ispu_max_before'] = df[ispu_cols_before].max(axis=1)
df['ispu_max_after']  = df[ispu_cols_after].max(axis=1)

def get_ispu_category(val):
    if pd.isna(val):
        return "Incomplete Data"
    elif val <= 50:
        return "Good"
    elif val <= 100:
        return "Moderate"
    elif val <= 200:
        return "Unhealthy"
    elif val <= 300:
        return "Very Unhealthy"
    else:
        return "Hazardous"

df['kategori_before'] = df['ispu_max_before'].apply(get_ispu_category)
df['kategori_after']  = df['ispu_max_after'].apply(get_ispu_category)

ISPU Category Distribution for 2010–2020

In [22]:
df_2010_2020 = df[(df['tahun'] >= 2010) & (df['tahun'] <= 2020)].copy()

dist_before = df_2010_2020['kategori_before'].value_counts(normalize=True) * 100
dist_after  = df_2010_2020['kategori_after'].value_counts(normalize=True) * 100

df_perbandingan = pd.DataFrame({
    'Before Reconstruction (%)': dist_before,
    'After Reconstruction (%)': dist_after
}).fillna(0).round(2)

print("========== ISPU CATEGORY DISTRIBUTION FOR 2010–2020 ==========")
print(df_perbandingan.to_string())

========== ISPU CATEGORY DISTRIBUTION FOR 2010–2020 ==========
                 Before Reconstruction (%)  After Reconstruction (%)
Good                                 36.46                      4.88
Incomplete Data                       2.74                      2.74
Moderate                             60.80                     17.55
Unhealthy                             0.00                     74.84


In [23]:

# Feature Importance Analysis — ElasticNet

import numpy as np
import pandas as pd

# Get the trained ElasticNet model
elastic_model = trained_models['ElasticNet Regression']

# Get the scaler used during training
elastic_scaler = scalers['ElasticNet Regression']

# Keep feature names in the same order used during training
features = ['pm10', 'so2', 'co', 'o3', 'no2']

# Get the model coefficients
coefficients = elastic_model.coef_

# Create a table of feature coefficients
importance_df = pd.DataFrame({
    'Feature': features,
    'Coefficient': coefficients,
    'Absolute Coefficient': np.abs(coefficients)
})

# Calculate each feature’s relative share of the absolute coefficients
total = importance_df['Absolute Coefficient'].sum()

importance_df['Relative Contribution (%)'] = (
    importance_df['Absolute Coefficient'] / total * 100
    if total != 0 else 0
)

importance_df = importance_df.sort_values(
    'Absolute Coefficient', ascending=False
)

print("ELASTICNET FEATURE ANALYSIS")
display(importance_df.round(4))


ELASTICNET FEATURE ANALYSIS


,Feature,Coefficient,Absolute Coefficient,Relative Contribution (%)
0,pm10,21.1401,21.1401,70.9883
4,no2,2.9847,2.9847,10.0226
3,o3,2.4202,2.4202,8.1269
1,so2,2.3364,2.3364,7.8457
2,co,0.8983,0.8983,3.0165


Reconstruct Historical PM2.5 Values

In [24]:

import pandas as pd
import numpy as np

# Make sure the date and year columns are available
df['tanggal'] = pd.to_datetime(df['tanggal'], errors='coerce')
df['tahun'] = df['tanggal'].dt.year

# Define the target and predictor features
target = 'pm25'
features = ['pm10', 'so2', 'co', 'o3', 'no2']

# Create a column for reconstructed PM2.5 values
df['pm25_reconstructed'] = df[target].copy()

# Select historical records from 2010–2020 where PM2.5 is missing
# and all predictor features are available
reconstruct_mask = (
    df['tahun'].between(2010, 2020)
    & df[target].isna()
    & df[features].notna().all(axis=1)
)

X_reconstruct = df.loc[reconstruct_mask, features]

if len(X_reconstruct) > 0:
    # Apply the matching scaler if the selected model requires scaling
    if best_config['scaled']:
        X_reconstruct_model = reconstruction_scaler.transform(
            X_reconstruct
        )
    else:
        X_reconstruct_model = X_reconstruct

    # Generate estimates using the best-performing model
    predictions = best_model.predict(X_reconstruct_model)

    # Store the estimates in the matching rows
    df.loc[reconstruct_mask, 'pm25_reconstructed'] = np.clip(
        predictions, 0, None
    )

# Display a summary of the reconstruction
historical_mask = df['tahun'].between(2010, 2020)

print("===== PM2.5 RECONSTRUCTION REPORT =====")
print("Model used:",
      best_model_name)
print("Total historical rows:",
      int(historical_mask.sum()))
print("Initially missing PM2.5 values:",
      int((historical_mask & df[target].isna()).sum()))
print("Rows successfully estimated:",
      int(reconstruct_mask.sum()))
print("PM2.5 values still missing:",
      int(df.loc[historical_mask,
                 'pm25_reconstructed'].isna().sum()))

print("\nSample reconstructed values:")
display(
    df.loc[reconstruct_mask,
           ['tanggal', target, 'pm25_reconstructed']].head()
)


===== PM2.5 RECONSTRUCTION REPORT =====
Model used: ElasticNet Regression
Total historical rows: 4018
Initially missing PM2.5 values: 4018
Rows successfully estimated: 3585
PM2.5 values still missing: 433

Sample reconstructed values:


,tanggal,pm25,pm25_reconstructed
0,2010-01-01,NaN,82.973353
1,2010-01-02,NaN,46.204690
2,2010-01-03,NaN,38.080723
3,2010-01-04,NaN,30.478441
4,2010-01-05,NaN,34.342551


Recalculate the ISPU Category Distribution

In [25]:

# Calculate sub-indices for the other pollutants
for p in ['pm10', 'so2', 'co', 'o3', 'no2']:
    df[f'ispu_{p}'] = df[p].apply(
        lambda x: calc_subindex(x, p)
    )

# Calculate PM2.5 sub-indices before and after reconstruction
df['ispu_pm25_raw'] = df['pm25'].apply(
    lambda x: calc_subindex(x, 'pm25')
)

df['ispu_pm25_rec'] = df['pm25_reconstructed'].apply(
    lambda x: calc_subindex(x, 'pm25')
)

# Select the sub-index columns used to calculate the maximum ISPU value
ispu_cols_before = [
    'ispu_pm10', 'ispu_so2', 'ispu_co',
    'ispu_o3', 'ispu_no2', 'ispu_pm25_raw'
]

ispu_cols_after = [
    'ispu_pm10', 'ispu_so2', 'ispu_co',
    'ispu_o3', 'ispu_no2', 'ispu_pm25_rec'
]

df['ispu_max_before'] = df[ispu_cols_before].max(axis=1)
df['ispu_max_after'] = df[ispu_cols_after].max(axis=1)

# Assign an ISPU category based on the maximum sub-index
def get_ispu_category(val):
    if pd.isna(val):
        return 'Incomplete Data'
    elif val <= 50:
        return 'Good'
    elif val <= 100:
        return 'Moderate'
    elif val <= 200:
        return 'Unhealthy'
    elif val <= 300:
        return 'Very Unhealthy'
    else:
        return 'Hazardous'

df['kategori_before'] = df['ispu_max_before'].apply(
    get_ispu_category
)
df['kategori_after'] = df['ispu_max_after'].apply(
    get_ispu_category
)

# Compare category distributions for 2010–2020
df_2010_2020 = df[
    df['tahun'].between(2010, 2020)
].copy()

categories = [
    'Good',
    'Moderate',
    'Unhealthy',
    'Very Unhealthy',
    'Hazardous',
    'Incomplete Data'
]

before = (
    df_2010_2020['kategori_before']
    .value_counts(normalize=True)
    .reindex(categories, fill_value=0) * 100
)

after = (
    df_2010_2020['kategori_after']
    .value_counts(normalize=True)
    .reindex(categories, fill_value=0) * 100
)

df_perbandingan = pd.DataFrame({
    'Before Reconstruction (%)': before,
    'After Reconstruction (%)': after
})

print("===== ISPU CATEGORY DISTRIBUTION (2010–2020) =====")
display(df_perbandingan.round(2))


===== ISPU CATEGORY DISTRIBUTION (2010–2020) =====


,Before Reconstruction (%),After Reconstruction (%)
Good,36.46,4.88
Moderate,60.80,17.55
Unhealthy,0.00,74.84
Very Unhealthy,0.00,0.00
Hazardous,0.00,0.00
Incomplete Data,2.74,2.74
